# Phase 3: GBIF Biodiversity Occurrence Data Collection & Validation
## Notebook 05: Coordinate Uncertainty and Record Quality Assessment

### Objective
Examine coordinate precision, investigate the `coordinateUncertaintyInMeters` attribute, evaluate the ~97% missing uncertainty phenomenon, and audit geospatial issue flags.

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

summary_path = os.path.join("..", "..", "data", "metadata", "phase_3_checks", "gbif_empirical_probe_summary.json")
with open(summary_path) as f:
    summary = json.load(f)

regions = ["bengaluru", "mysuru", "nilgiri_bandipur"]
region_names = {"bengaluru": "Bengaluru", "mysuru": "Mysuru", "nilgiri_bandipur": "Nilgiri-Bandipur"}

### 1. Empirical Breakdown of Coordinate Uncertainty
We calculate the distribution of positional uncertainty across each study landscape.

In [2]:
unc_rows = []
for rid in regions:
    d = summary[rid]
    total = d["polygon_count"]
    le100 = d["uncertainty_le_100m"]
    le1000 = d["uncertainty_le_1000m"]
    gt1000 = d["uncertainty_gt_1000m"]
    stated = d["uncertainty_any_stated"]
    missing = d["uncertainty_missing"]
    
    unc_rows.append({
        "Landscape": region_names[rid],
        "Total Records": total,
        "Missing Uncertainty": missing,
        "Missing Share (%)": round(missing / total * 100, 2),
        "Uncertainty ≤ 100m": le100,
        "≤ 100m Share (%)": round(le100 / total * 100, 2),
        "Uncertainty ≤ 1km": le1000,
        "≤ 1km Share (%)": round(le1000 / total * 100, 2),
        "Uncertainty > 1km": gt1000,
        "> 1km Share (%)": round(gt1000 / total * 100, 2)
    })

df_unc = pd.DataFrame(unc_rows)
print("=== Coordinate Uncertainty Breakdown ===")
print(df_unc.to_string(index=False))

=== Coordinate Uncertainty Breakdown ===
       Landscape  Total Records  Missing Uncertainty  Missing Share (%)  Uncertainty ≤ 100m  ≤ 100m Share (%)  Uncertainty ≤ 1km  ≤ 1km Share (%)  Uncertainty > 1km  > 1km Share (%)
       Bengaluru        3558771              3457634              97.16               43462              1.22              87960             2.47              13177             0.37
          Mysuru         937184               914539              97.58                7896              0.84              18048             1.93               4597             0.49
Nilgiri-Bandipur         240252               231411              96.32                3394              1.41               5047             2.10               3794             1.58


### 2. The Missing Uncertainty Phenomenon
Between **96.3% and 97.6% of records** omit numerical values for `coordinateUncertaintyInMeters`. 
This occurs because standard eBird checklist submissions do not require observers to supply a GPS error radius. 
**Crucial Filtering Decision**: We **cannot** simply filter by `coordinateUncertaintyInMeters <= 1000`, as doing so would discard >96% of genuine empirical bird observations. Missing uncertainty must be preserved as its own category rather than treated as inaccurate.

In [3]:
# Geospatial Issues Audit
geo_rows = []
for rid in regions:
    d = summary[rid]
    total = d["polygon_count"]
    issues_dict = d["geospatial_issues"]
    clean_recs = issues_dict.get("0", total)
    flagged_recs = issues_dict.get("1", 0)
    
    geo_rows.append({
        "Landscape": region_names[rid],
        "Clean Records (No Issues)": clean_recs,
        "Flagged Geospatial Issues": flagged_recs,
        "Flagged Share (%)": round(flagged_recs / total * 100, 5)
    })

df_geo = pd.DataFrame(geo_rows)
print("=== Geospatial Quality Issue Audit ===")
print(df_geo.to_string(index=False))

=== Geospatial Quality Issue Audit ===
       Landscape  Clean Records (No Issues)  Flagged Geospatial Issues  Flagged Share (%)
       Bengaluru                    3558760                         11            0.00031
          Mysuru                     937182                          2            0.00021
Nilgiri-Bandipur                     240247                          5            0.00208
